# 04 — Curriculum RL-PLUS v2: Safe Medical Triage

## Changes from v1 (based on training log analysis)

| # | Change | Why |
|---|--------|-----|
| 1 | **Gradual reward** (exploration-friendly, not binary 0/1) | Partial credit → diverse advantages |
| 2 | **Replay** (cumulative rows across stages) | Prevents catastrophic forgetting |
| 3 | **400 steps/stage** (was 200) | Stage 1 didn't plateau at 200 |
| 4 | **λ=1.0 everywhere** (was 1.5→1.0→0.5) | Hard cases need MORE gold signal |
| 5 | **Temperature=0.8 everywhere** (was 0.8→0.7→0.6) | Hard needs MORE exploration |
| 6 | **SFT 50 steps** (was 200) | Loss hit 0 at step 25; less overfit → more RL room |
| 7 | **group_size=16 on easy/medium** | More diversity → reward_std>0 more often |
| 8 | **Key order fix** (name before args) | Gold now matches prompt schema format |
| 9 | **No reward inversion** (trying > nothing) | Old: empty=1.0 > tried=0.45; new: tried=0.55 > empty=0.0 |
| 10 | **Per-row gold_rewards** | Each case uses its own Oracle reward, not batch[0] |
| 11 | **SFT trains EOS** | Model learns to stop after </ACTIONS> |
| 12 | **max_new_tokens 1536→512** | Gold ~150 tokens; saves ~20% training time |

**Hardware:** RTX 6000 Pro Blackwell (96 GB VRAM)

## 1. Setup & control flags

In [ ]:
import os, sys, gc, json, time
import warnings
warnings.filterwarnings("ignore", message=".*attention mask.*deprecated.*")

try:
    import comet_ml  # pip install comet-ml
except ImportError:
    print("comet_ml not installed — training will work but no online dashboard")
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# ─────────────────────────────────────────────
# CONTROL FLAGS
# ─────────────────────────────────────────────
RUN_GENERATE_DATA   = True
RUN_BUILD_GOLD_ROWS = True

RUN_SFT             = True
FORCE_SFT           = False

RUN_STAGE1_EASY     = True
FORCE_STAGE1        = False

RUN_STAGE2_MEDIUM   = True
FORCE_STAGE2        = True

RUN_STAGE3_HARD     = True
FORCE_STAGE3        = False

RUN_EXPORT          = True
FORCE_EXPORT        = False
# ─────────────────────────────────────────────

TRAIN_MODEL = "Qwen/Qwen3-8B"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "runs" / "curriculum_rl_plus_8b"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_ROOT / "data"

SFT_ADAPTER_DIR    = str(ARTIFACTS_DIR / "sft_adapter")
STAGE1_ADAPTER_DIR = str(ARTIFACTS_DIR / "stage1_easy" / "adapter")
STAGE2_ADAPTER_DIR = str(ARTIFACTS_DIR / "stage2_medium" / "adapter")
STAGE3_ADAPTER_DIR = str(ARTIFACTS_DIR / "stage3_hard" / "adapter")
MERGED_MODEL_DIR   = str(ARTIFACTS_DIR / "merged_16bit")
MANIFEST_FILE      = str(ARTIFACTS_DIR / "manifest.json")
GOLD_ROWS_FILE     = str(ARTIFACTS_DIR / "gold_train_rows.jsonl")

print(f"ARTIFACTS = {ARTIFACTS_DIR}")
print(f"FORCE: SFT={FORCE_SFT} S1={FORCE_STAGE1} S2={FORCE_STAGE2} S3={FORCE_STAGE3}")

ARTIFACTS = /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b
FORCE: SFT=False S1=False S2=False S3=False


## 2. Generate datasets (idempotent)

In [2]:
from triage import generate_train_val_datasets, generate_eval_datasets, run_oracle_on_dataset

train_path = DATA_DIR / "train.jsonl"
if RUN_GENERATE_DATA and not train_path.exists():
    generate_train_val_datasets(out_dir=DATA_DIR, train_size=5000, val_size=500, seed=13, difficulties=range(1, 11))
print(f"Train: {train_path} ({'exists' if train_path.exists() else 'MISSING'})")

eval_dir = DATA_DIR / "eval"
if RUN_GENERATE_DATA and not (eval_dir / "eval_d1.jsonl").exists():
    generate_eval_datasets(out_dir=eval_dir, size_per_difficulty=100, seed=101, difficulties=range(1, 11))

eval_paths = {d: eval_dir / f"eval_d{d}.jsonl" for d in range(1, 11) if (eval_dir / f"eval_d{d}.jsonl").exists()}
print(f"Eval: {len(eval_paths)} buckets")

oracle = run_oracle_on_dataset(eval_paths[1])
assert oracle.summary["success_rate"] == 1.0
print(f"Oracle d1: {oracle.summary['success_rate']:.0%} ✓")

Train: /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/data/train.jsonl (exists)
Eval: 10 buckets


oracle:eval_d1.jsonl:   0%|          | 0/100 [00:00<?, ?it/s]

[INFO] Oracle summary: {"num_episodes": 100, "success_rate": 1.0, "avg_total_reward": 0.7694999999999999, "avg_steps": 5.06, "avg_tool_calls": 4.31, "avg_policy_violations": 0.0, "avg_critical_policy_violations": 0.0, "invalid_action_rate": 0.0, "undertriage_rate": 0.0, "over_escalation_rate": 0.0, "hallucination_rate": 0.0, "confirmation_violation_rate": 0.0, "duplicate_question_rate": 0.0, "irrelevant_question_rate": 0.0, "avg_evidence_coverage": 1.0, "failure_reasons": {}, "num_cases": 100, "successes": 100}
Oracle d1: 100% ✓


## 3. Build gold rows with Oracle (idempotent)

In [3]:
from triage.io_utils import read_dataset
from runtimes.train_unsloth.triage_rl_plus_compat import build_triage_train_rows

gold_rows_path = Path(GOLD_ROWS_FILE)

need_rebuild = False
if RUN_BUILD_GOLD_ROWS and not gold_rows_path.exists():
    need_rebuild = True
elif gold_rows_path.exists():
    with gold_rows_path.open() as f:
        first_line = f.readline().strip()
    if first_line:
        row0 = json.loads(first_line)
        # Detect stale cache: missing gold_reward, or old key order, or wrong reward scale
        if "gold_reward" not in row0:
            print("⚠ Stale cache: missing gold_reward — rebuilding")
            need_rebuild = True
        elif row0.get("gold_reward", 0) > 1.5:
            print("⚠ Stale cache: old reward scale (shifted verifier) — rebuilding")
            need_rebuild = True
        elif '"args"' in (row0.get("gold_completion", "")[:100]) and \
             row0.get("gold_completion", "").find('"args"') < row0.get("gold_completion", "").find('"name"'):
            print("⚠ Stale cache: old key order (args before name) — rebuilding")
            need_rebuild = True

if need_rebuild:
    cases = read_dataset(train_path)
    all_rows = build_triage_train_rows(cases, progress=True)
    gold_rows_path.parent.mkdir(parents=True, exist_ok=True)
    with gold_rows_path.open("w") as f:
        for r in all_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(all_rows)} gold rows (v5.5: correct key order + exploration reward)")
else:
    all_rows = []
    with gold_rows_path.open() as f:
        for line in f:
            if line.strip():
                all_rows.append(json.loads(line))
    print(f"Loaded {len(all_rows)} cached gold rows")

print(f"gold_reward example: {all_rows[0].get('gold_reward', 'MISSING')}")
print(f"gold key order: {'name first ✓' if all_rows[0]['gold_completion'].find('"name"') < all_rows[0]['gold_completion'].find('"args"') else 'WRONG ORDER'}")

rows_easy   = [r for r in all_rows if r["difficulty"] <= 3]
rows_medium = [r for r in all_rows if 4 <= r["difficulty"] <= 7]
rows_hard   = [r for r in all_rows if r["difficulty"] >= 8]
print(f"Easy: {len(rows_easy)}  Medium: {len(rows_medium)}  Hard: {len(rows_hard)}")

⚠ Stale cache: old reward scale (shifted verifier) — rebuilding


building train rows:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved 5000 gold rows (v5.5: correct key order + exploration reward)
gold_reward example: 1.8
gold key order: name first ✓
Easy: 1500  Medium: 2000  Hard: 1500


In [4]:
def adapter_exists(path: str) -> bool:
    p = Path(path)
    return p.is_dir() and (
        (p / "adapter_model.safetensors").exists()
        or (p / "adapter_model.bin").exists()
        or (p / "model.safetensors").exists()
    )

def merged_exists(path: str) -> bool:
    p = Path(path)
    return p.is_dir() and any(p.glob("*.safetensors"))

for name, path in [("SFT", SFT_ADAPTER_DIR), ("Stage1", STAGE1_ADAPTER_DIR),
                    ("Stage2", STAGE2_ADAPTER_DIR), ("Stage3", STAGE3_ADAPTER_DIR)]:
    print(f"  {name}: {'EXISTS' if adapter_exists(path) else '—'}")

  SFT: EXISTS
  Stage1: EXISTS
  Stage2: —
  Stage3: —


In [5]:
from runtimes.train_unsloth.triage_rl_plus_compat import (
    make_triage_verify_fn, make_triage_reward_fn, make_triage_gold_fn,
)

verify_fn = make_triage_verify_fn()   # bool: for correct_rate logging
reward_fn = make_triage_reward_fn()   # float: gradual reward for GRPO advantage
gold_fn   = make_triage_gold_fn()     # str: Oracle completion

# Quick test
test_row = all_rows[0]
print(f"Gold:    reward={reward_fn(test_row, test_row['gold_completion']):.2f}  success={verify_fn(test_row, test_row['gold_completion'])}")
print(f"Garbage: reward={reward_fn(test_row, 'no actions here'):.2f}  success={verify_fn(test_row, 'no actions here')}")

Gold:    reward=1.80  success=True
Garbage: reward=0.00  success=False


## 4. Stage 0: SFT (50 steps)

v1 used 200 steps → loss was 0 after step 25 → overfitting → model stuck near SFT point.
Now 50 steps: model learns format but retains exploration capacity.

In [6]:
if RUN_SFT and (FORCE_SFT or not adapter_exists(SFT_ADAPTER_DIR)):
    from runtimes.train_unsloth.sft_trainer import SFTConfig, train_sft_lora

    sft_cfg = SFTConfig(
        model_name=TRAIN_MODEL,
        output_dir=SFT_ADAPTER_DIR,
        max_steps=50,              # ← was 200; loss=0 at step 25
        batch_size=4,
        lr=1e-4,
        max_seq_len=2048,
        grad_accum_steps=2,
        load_in_4bit=False,
        torch_dtype="bf16",
        device="cuda",
        lora_r=64,
        lora_alpha=128,
        use_unsloth=True,
        unsloth_max_seq_length=2048,
        unsloth_disable_compile=True,
        clear_unsloth_cache=True,
        enable_gradient_checkpointing=True,
    )

    t0 = time.time()
    sft_dir, sft_history = train_sft_lora(train_rows=all_rows, cfg=sft_cfg)
    print(f"SFT done in {(time.time()-t0)/60:.1f} min, final loss: {sft_history[-1]['loss']:.4f}")

    with open(str(ARTIFACTS_DIR / "sft_history.json"), "w") as f:
        json.dump(sft_history, f)
    gc.collect()
    if __import__('torch').cuda.is_available(): __import__('torch').cuda.empty_cache()
else:
    print(f"SFT: {'exists' if adapter_exists(SFT_ADAPTER_DIR) else 'disabled'}")

SFT: exists


## 5. Curriculum RL-PLUS stages

**Key v2 changes:**
- **Replay**: each stage includes all previous difficulty rows
- **λ=1.0 everywhere**: hard cases need gold signal most
- **T=0.8 everywhere**: hard cases need exploration most
- **400 steps**: Stage 1 was still learning at step 200
- **group_size=16 on easy/medium**: more reward diversity per group
- **Gradual reward**: `reward_fn` replaces binary 0/1

In [ ]:
from runtimes.train_unsloth.rl_plus_trainer import RLPlusConfig, RLPlusTrainer

# Qwen3-8B speed config: group=4, no grad checkpoint, higher LR, fewer steps

STAGES = [
    {
        "name": "stage1_easy",
        "flag": RUN_STAGE1_EASY, "force": FORCE_STAGE1,
        "rows": rows_easy,
        "adapter_dir": STAGE1_ADAPTER_DIR,
        "lambda_ext": 1.0,
        "temperature": 0.8,
        "max_steps": 150,
        "lr": 1e-5,
        "batch_size_prompts": 8,
        "group_size": 4,
        "grad_accum_steps": 1,
    },
    {
        "name": "stage2_medium",
        "flag": RUN_STAGE2_MEDIUM, "force": FORCE_STAGE2,
        "rows": rows_easy + rows_medium,
        "adapter_dir": STAGE2_ADAPTER_DIR,
        "lambda_ext": 1.0,
        "temperature": 0.8,
        "max_steps": 150,
        "lr": 1e-6,
        "batch_size_prompts": 8,
        "group_size": 4,
        "grad_accum_steps": 1,
    },
    {
        "name": "stage3_hard",
        "flag": RUN_STAGE3_HARD, "force": FORCE_STAGE3,
        "rows": rows_easy + rows_medium + rows_hard,
        "adapter_dir": STAGE3_ADAPTER_DIR,
        "lambda_ext": 1.0,
        "temperature": 0.8,
        "max_steps": 150,
        "lr": 3e-6,
        "batch_size_prompts": 8,
        "group_size": 4,
        "grad_accum_steps": 1,
    },
]

# ─── Comet.ml config ───
COMET_PROJECT   = "triage-rl-plus"
COMET_WORKSPACE = None
# ────────────────────────

BASE_RL_CFG = dict(
    model_name=TRAIN_MODEL,
    seed=42,
    max_grad_norm=0.5,
    max_new_tokens=256,
    top_p=0.95,
    beta_kl=0.01,
    clip_eps=0.2,
    gamma_focal=2.0,
    r_m_max=10.0,
    format_bonus=0.0,
    load_in_4bit=False,
    torch_dtype="bf16",
    device="cuda",
    ref_use_disable_adapter=True,
    loss_chunk_size=4,
    use_lora=True,
    lora_r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    use_unsloth=True,
    unsloth_max_seq_length=1024,
    unsloth_disable_compile=True,
    clear_unsloth_cache=True,
    enable_gradient_checkpointing=True,  # OFF — 46 GB headroom, skip recomputation
    print_rollout_samples_every=50,
    print_rollout_samples_n=2,
    cleanup_every_steps=50,
    save_every_steps=30,
    comet_project=COMET_PROJECT,
    comet_workspace=COMET_WORKSPACE,
    comet_log_parameters=True,
)

adapter_chain = [SFT_ADAPTER_DIR, STAGE1_ADAPTER_DIR, STAGE2_ADAPTER_DIR, STAGE3_ADAPTER_DIR]
current_adapter = SFT_ADAPTER_DIR
for ad in adapter_chain:
    if adapter_exists(ad):
        current_adapter = ad

print(f"Model: {TRAIN_MODEL}")
print(f"Speed: group=4, no grad_ckpt, max_new=256, 150 steps/stage")
print(f"Comet: {COMET_PROJECT or 'disabled'}")
print(f"Resuming from: {current_adapter}")
for s in STAGES:
    status = "EXISTS" if adapter_exists(s["adapter_dir"]) else ("RUN" if s["flag"] else "SKIP")
    seqs = s["batch_size_prompts"] * s["group_size"]
    print(f"  {s['name']:15s}  {len(s['rows']):>5d} rows  {s['batch_size_prompts']}x{s['group_size']}={seqs:>3d} seq  lr={s['lr']:.0e}  {s['max_steps']} steps  | {status}")

Model: Qwen/Qwen3-8B
Speed: group=4, no grad_ckpt, max_new=256, 150 steps/stage
Comet: triage-rl-plus
Resuming from: /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b/stage1_easy/adapter
  stage1_easy       1500 rows  8x4= 32 seq  lr=1e-05  150 steps  | EXISTS
  stage2_medium     3500 rows  8x4= 32 seq  lr=5e-06  150 steps  | RUN
  stage3_hard       5000 rows  8x4= 32 seq  lr=3e-06  150 steps  | RUN


## 6. Run curriculum training

In [ ]:
all_history = {}

for s in STAGES:
    hist_path = ARTIFACTS_DIR / s["name"] / "history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            all_history[s["name"]] = json.load(f)
        print(f"Loaded cached history: {s['name']} ({len(all_history[s['name']])} steps)")




for stage_idx, stage in enumerate(STAGES):
    name = stage["name"]
    should_run = stage["flag"] and (stage.get("force", False) or not adapter_exists(stage["adapter_dir"]))

    if not should_run:
        if adapter_exists(stage["adapter_dir"]):
            current_adapter = stage["adapter_dir"]
            current_adapter = str(ARTIFACTS_DIR / "stage1_easy" / "adapter" / "checkpoint-60")
            print(f"[{name}] exists — skipping")
        else:
            print(f"[{name}] disabled")
        continue

    print(f"\n{'='*60}")
    print(f"STAGE {stage_idx+1}: {name}")
    print(f"  rows={len(stage['rows'])}  λ={stage['lambda_ext']}  T={stage['temperature']}  batch={stage['batch_size_prompts']}×{stage['group_size']}×{stage['grad_accum_steps']}accum  peak={stage['batch_size_prompts']*stage['group_size']}seq")
    print(f"  init_adapter={current_adapter}")
    print(f"{'='*60}")

    cfg = RLPlusConfig(
        **BASE_RL_CFG,
        output_dir=stage["adapter_dir"],
        init_adapter_path=current_adapter,
        ref_adapter_path=current_adapter,
        lambda_ext=stage["lambda_ext"],
        temperature=stage["temperature"],
        lr=stage["lr"],
        max_steps=stage["max_steps"],
        batch_size_prompts=stage["batch_size_prompts"],
        group_size=stage["group_size"],
        grad_accum_steps=stage["grad_accum_steps"],
        comet_experiment_name=f"{TRAIN_MODEL.split('/')[-1]}_{name}",  # separate comet experiment per stage
    )

    trainer = RLPlusTrainer(cfg)
    t0 = time.time()

    history = trainer.train(
        train_rows=stage["rows"],
        env_verify_fn=verify_fn,
        reward_fn=reward_fn,                # ← gradual reward
        make_gold_completion_fn=gold_fn,
        log_every=10,
        eval_every=0,
    )

    elapsed = time.time() - t0
    print(f"[{name}] Done in {elapsed/60:.1f} min")

    saved_dir = trainer.save()
    # Prefer checkpoint-best if it exists
    best_dir = str(Path(stage["adapter_dir"]) / "checkpoint-best")
    current_adapter = best_dir if Path(best_dir).exists() else saved_dir
    print(f"  Next stage will use: {current_adapter}")
    all_history[name] = history

    hist_dir = ARTIFACTS_DIR / name
    hist_dir.mkdir(parents=True, exist_ok=True)
    with open(str(hist_dir / "history.json"), "w") as f:
        json.dump(history, f)

    trainer.cleanup(move_models_to_cpu=True)
    del trainer
    gc.collect()
    if __import__('torch').cuda.is_available(): __import__('torch').cuda.empty_cache()

FINAL_ADAPTER = current_adapter
print(f"\nFinal adapter: {FINAL_ADAPTER}")

Loaded cached history: stage1_easy (90 steps)
[stage1_easy] exists — skipping

STAGE 2: stage2_medium
  rows=3500  λ=1.0  T=0.8  batch=8×4×1accum  peak=32seq
  init_adapter=/home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b/stage1_easy/adapter/checkpoint-60
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 95.592 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enable

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/yaroslav-pankratov/triage-rl-plus/cd78c86cecf3423fa73867503945d487

COMET INFO: Couldn't find a Git repository in '/home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


GRPO/RL-PLUS train:   0%|          | 0/150 [00:00<?, ?step/s]

--- Logging error ---
Traceback (most recent call last):
  File "/home/yaros/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/yaros/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/yaros/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/yaros/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "/home/yaros/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/li

step 10/150 | loss=-0.0157 int=0.0235 ext=-0.0392 kl=1.3379 r_m=1.108 r=0.943±0.481 acc=0.219 parse=1.000 fmt=0.969 mae=0.00 len=170.1 grad=1.58 lr=5.00e-06 t=44.39s
step 20/150 | loss=-0.0067 int=0.0837 ext=-0.0904 kl=1.5896 r_m=1.082 r=0.751±0.226 acc=0.031 parse=1.000 fmt=0.781 mae=0.00 len=169.0 grad=4.20 lr=5.00e-06 t=44.39s
step 30/150 | loss=0.0202 int=0.1158 ext=-0.0957 kl=1.5694 r_m=1.071 r=0.781±0.285 acc=0.062 parse=1.000 fmt=0.469 mae=0.00 len=209.7 grad=9.04 lr=5.00e-06 t=45.01s
  [checkpoint] saved → /home/yaros/DS-Mag/AI-SelectedTopics/W4/safe_triage_env_project_w3_integrated/artifacts/runs/curriculum_rl_plus_8b/stage2_medium/adapter/checkpoint-30
step 40/150 | loss=-0.0076 int=0.0714 ext=-0.0791 kl=1.5082 r_m=1.059 r=0.771±0.353 acc=0.094 parse=1.000 fmt=0.344 mae=0.00 len=215.9 grad=8.26 lr=5.00e-06 t=44.82s
step 50/150 | loss=-0.0439 int=0.0699 ext=-0.1138 kl=1.1563 r_m=1.064 r=0.689±0.110 acc=0.000 parse=1.000 fmt=0.281 mae=0.00 len=224.0 grad=14.74 lr=5.00e-06 t=44.

## 7. Training curves

In [ ]:
import pandas as pd

if all_history:
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(2, 3, figsize=(18, 8))
        metrics_to_plot = ["reward_int_mean", "correct_rate", "loss", "loss_ext", "kl", "len_mean"]
        for ax, metric in zip(axes.flat, metrics_to_plot):
            offset = 0
            for name, hist in all_history.items():
                df = pd.DataFrame(hist)
                if metric in df.columns:
                    # Smoothed
                    vals = df[metric].rolling(window=20, min_periods=1).mean()
                    ax.plot(df["step"] + offset, vals, label=name, alpha=0.8)
                offset += len(hist)
            ax.set_ylabel(metric); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
        plt.suptitle("Curriculum RL-PLUS v2 (smoothed)", y=1.01)
        plt.tight_layout()
        plt.savefig(str(ARTIFACTS_DIR / "curriculum_curves_v2.png"), dpi=150, bbox_inches="tight")
        plt.show()
    except ImportError:
        for name, hist in all_history.items():
            df = pd.DataFrame(hist)
            if "reward_int_mean" in df.columns:
                r = df["reward_int_mean"]
                print(f"{name}: reward {r.iloc[0]:.3f} → {r.iloc[-1]:.3f}")
else:
    print("No history")

## 8. Export merged model

In [ ]:
if RUN_EXPORT and (FORCE_EXPORT or not merged_exists(MERGED_MODEL_DIR)):
    from runtimes.train_unsloth import export_merged_model_for_vllm
    export_result = export_merged_model_for_vllm(
        FINAL_ADAPTER, base_model=TRAIN_MODEL, run_name="curriculum_rl_plus_8b",
        export_dir=MERGED_MODEL_DIR,
        manifest=MANIFEST_FILE if Path(MANIFEST_FILE).exists() else None,
        max_seq_length=4096,
    )
    print(f"Merged: {MERGED_MODEL_DIR}")
    print(f"Manifest: {export_result['manifest_path']}")
elif merged_exists(MERGED_MODEL_DIR):
    print(f"Merged exists: {MERGED_MODEL_DIR}")
else:
    print("Export disabled")

## 9. v1 vs v2 changes summary

### Why binary reward killed learning

With binary 0/1:
```
Trajectory A: covers 80% evidence, right disposition, misses confirmation → reward = 0
Trajectory B: immediate finish, wrong disposition                         → reward = 0
GRPO advantage(A) = advantage(B) = 0.   No learning.
```

With gradual reward (shifted verifier total_reward):
```
Trajectory A: covers 80% evidence, right disposition → reward ≈ 1.8
Trajectory B: immediate finish, wrong disposition    → reward ≈ 0.5
GRPO advantage(A) > advantage(B).   Model learns to prefer A.
```

### Why replay prevents forgetting

v1 Stage 3 saw only d8-d10 → forgot d1-d3 patterns.
v2 Stage 3 sees d1+d2+...+d10 → random batch mixes easy and hard.
Easy cases provide stable positive reward → anchor for learning.

### Expected improvements

| Metric | v1 | v2 expected |
|--------|-----|-------------|
| Dead steps (reward_std=0) | 2-26% | <5% (gradual rewards are always diverse) |
| Stage 1 correct_rate (end) | 31% | Higher (400 steps, group=16) |
| Stage 3 reward trend | −0.068 (declining) | Stable or rising (replay + constant λ) |